# 01 · Head-of-line blocking: single vs dual endpoint

Hot-class latency under a cold bulk stream, one endpoint vs two, on native libibverbs and libfabric.

In [ ]:
MODE = "local"            # "local": smoke runs on this machine (tcp / tcp;ofi_rxm / shm providers)
                          # "cluster": this host is the client, the server runs on PEER over ssh
PEER = "cerberus"         # ssh host for the server side (cluster mode)
PEER_IP = "192.168.1.242" # server address on the RDMA netdev; default taken from benchmark_dual_qp_remote.py, check it
REMOTE_REPO = None        # repo path on PEER if it differs from this host
DOMAIN = "rxe0"           # RDMA device / libfabric domain on both nodes
SV_GID_INDEX = None       # set if the auto-picked GID (first IPv4-mapped) is wrong, e.g. "1"
HOT_CPU = None            # pin the hot loop (cluster: pick a core away from the NIC IRQs)
COLD_CPU = None           # pin the cold stream thread
ITERS = None              # None: 2000 local, 20000 cluster
RESULTS_SUBDIR = "fabric"


In [ ]:
import os, sys, json, socket, subprocess, time
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "middleware").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
HOST = socket.gethostname()
STAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
RESULTS = (ROOT / "benchmarks" / "results" / RESULTS_SUBDIR) if MODE == "cluster" else Path("/tmp/scuffed-local") / RESULTS_SUBDIR
RESULTS.mkdir(parents=True, exist_ok=True)
def save(name, obj):
    p = RESULTS / f"{HOST}_{name}.json"
    obj = {"host": HOST, "utc": STAMP, "mode": MODE, **obj}
    p.write_text(json.dumps(obj, indent=2, default=str))
    print("wrote", p)
    return p
def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=ROOT)
    print(r.stdout[-4000:], r.stderr[-2000:])
    if check and r.returncode:
        raise RuntimeError(f"{cmd} -> {r.returncode}")
    return r
print("repo:", ROOT, "| host:", HOST, "| mode:", MODE, "| results ->", RESULTS)

**Claim under test:** a small latency-critical transfer that shares a queue pair (endpoint) with bulk
transfers waits behind them (head-of-line blocking); splitting hot and cold traffic onto separate QPs /
endpoints removes that wait.

**Method.** `sf_bench -m hol` / `sv_bench -m hol`: the client streams bulk RDMA writes on the cold class
(window `-W`, size `--cold-size`) while the hot class ping-pongs small writes with immediate data. Hot
round-trip percentiles are measured with the cold stream off and on, with both classes on one endpoint
(`-s`, the baseline) and on separate endpoints (the dual pool). Same binary, same payloads, same timing for
native libibverbs and libfabric; only the library under it changes.

**Read it as:** the claim holds on a stack if dual-endpoint hot p99 under cold load stays near the unloaded
p99 while single-endpoint hot p99 grows with cold message size. If both grow, the bottleneck is below the
QP (the NIC or rxe's single kernel thread), which is itself a result worth reporting.

**Retraction.** The 8 µs cross-node figure from Update 4 (`benchmarks/results/dual_qp_remote_benchmark.json`)
came from a pool that did not check `wc.status`; 8 MB writes "completed" in about 8 µs, which 10GbE cannot do.
The pool now raises on error and flush completions. The last section re-runs that benchmark.

In [ ]:
from benchmarks.fabric import runner
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
CLUSTER = MODE == "cluster"
N = ITERS or (20000 if CLUSTER else 2000)
W = max(100, N // 20)
PAIR = dict(peer=PEER, peer_ip=PEER_IP, remote_repo=REMOTE_REPO) if CLUSTER else {}
ENV = {"SV_GID_INDEX": str(SV_GID_INDEX)} if SV_GID_INDEX is not None else {}
STACKS = runner.stacks(MODE)
if CLUSTER:
    for st in STACKS:
        if st["tool"] == "sf_bench":
            st["args"] += ["-d", DOMAIN]
        else:
            ENV.setdefault("SV_IB_PORT", "1")
PIN = (["--hot-cpu", str(HOT_CPU)] if HOT_CPU is not None else []) + (["--cold-cpu", str(COLD_CPU)] if COLD_CPU is not None else [])
print(runner.build(remote=PEER if CLUSTER else None, remote_repo=REMOTE_REPO))
for st in STACKS: print(st["label"], st["tool"], st["args"])

In [ ]:
HOT_SIZES = [64, 4096]
COLD_SIZES = [256 * 1024, 1 << 20, 4 << 20] if CLUSTER else [256 * 1024, 1 << 20]
raw, rows = [], []
for st in STACKS:
    for single in (False, True):
        tag = "single EP" if single else "dual EP"
        for hot in HOT_SIZES:
            base = st["args"] + ["-m", "hol", "-n", str(N), "-w", str(W), "--hot-size", str(hot), "-W", "8"] + PIN + (["-s"] if single else [])
            d = runner.try_pair(st["tool"], base + ["--no-cold"], env=ENV, **PAIR)
            raw.append(d); rows.append(runner.rows_hol(d, f"{st['label']} | {tag}"))
            for cs in COLD_SIZES:
                d = runner.try_pair(st["tool"], base + ["--cold-size", str(cs)], env=ENV, **PAIR)
                raw.append(d); rows.append(runner.rows_hol(d, f"{st['label']} | {tag}"))
df = pd.DataFrame(rows)
df

In [ ]:
save("hol_blocking", {"rows": rows, "raw": raw})
ok = df[df.get("error").isna()] if "error" in df else df
fig, axes = plt.subplots(1, len(HOT_SIZES), figsize=(7 * len(HOT_SIZES), 4.5), squeeze=False)
for ax, hot in zip(axes[0], HOT_SIZES):
    sub = ok[ok.hot_size == hot]
    for lab, g in sub.groupby("label"):
        g = g.sort_values(["cold_on", "cold_size"])
        x = ["off" if not c else f"{s >> 10} KiB" for c, s in zip(g.cold_on, g.cold_size)]
        ax.plot(x, g.hot_p99_us, marker="o", label=lab)
    ax.set_yscale("log"); ax.set_xlabel("cold stream"); ax.set_ylabel("hot RTT p99 (us)")
    ax.set_title(f"hot {hot} B"); ax.grid(alpha=.3)
axes[0][0].legend(fontsize=7)
plt.tight_layout()

In [ ]:
# HOL penalty: loaded p99 / unloaded p99, single vs dual, per stack and cold size
pen = []
for lab, g in ok.groupby("label"):
    for hot, gh in g.groupby("hot_size"):
        un = gh[gh.cold_on == 0].hot_p99_us
        if un.empty: continue
        for _, r in gh[gh.cold_on == 1].iterrows():
            pen.append({"label": lab, "hot_size": hot, "cold_size": r.cold_size, "p99_ratio": r.hot_p99_us / un.iloc[0], "cold_MBps": r.cold_MBps})
pd.DataFrame(pen)

## Re-run of the Update 4 pyverbs benchmark (cluster only)

`benchmarks/benchmark_dual_qp_remote.py` drives the Python pool. With the completion-status fix, a failed
write now raises instead of looking fast. Run it to replace the retracted number.

In [ ]:
if CLUSTER:
    out = RESULTS / f"{HOST}_dual_qp_remote_rerun.json"
    print("on the server node (PEER):")
    print(f"  python3 benchmarks/benchmark_dual_qp_remote.py --role server --device {DOMAIN}")
    r = sh(f"python3 benchmarks/benchmark_dual_qp_remote.py --role client --host {PEER_IP} --device {DOMAIN} --output {out}", check=False)
else:
    print("cluster mode only (needs pyverbs and an RDMA device on both nodes)")